# PR Governance Agent - Live Demo

**No API keys required to trigger the agent.** A GitHub token is only needed in Step 5 to read results from a private repo.

1. Fill in `REPO` and `PR_NUMBER` in Cell 1
2. Click **Runtime > Run all**
3. Watch live results in Cell 5

## Step 1: Configure

In [ ]:
from getpass import getpass

# Required: set your target repo and PR number
REPO = "lmudu2/risk-analyzer-poc"
PR_NUMBER = "155"  # <-- set to an open PR number

# Only needed to read results from a private repo (leave blank for public repos)
GITHUB_TOKEN = getpass("GitHub PAT (for reading results from private repo, or press Enter to skip): ")

GATEWAY_URL = "https://zrfcfcrgrwublxw22es6gvuidq0ckmpm.lambda-url.us-east-1.on.aws/"
print(f"Target: {REPO} PR #{PR_NUMBER}")
print(f"Agent:  {GATEWAY_URL}")
print(f"Auth:   {'token provided' if GITHUB_TOKEN else 'no token (public repo mode)'}")

## Step 2: Trigger Real Risk Analysis

In [ ]:
import urllib.request, json

payload = {
    "action": "opened",
    "pull_request": {"number": int(PR_NUMBER), "title": f"Demo PR #{PR_NUMBER}", "user": {"login": "demo-presenter", "type": "User"}},
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(payload).encode(),
    headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"Agent triggered (HTTP {res.getcode()}) - analyzing PR #{PR_NUMBER}...")
print(f"Watch GitHub: https://github.com/{REPO}/pull/{PR_NUMBER}")

## Step 3: Simulate HIGH RISK (Schema Change)

In [ ]:
high_risk = {
    "action": "opened",
    "pull_request": {"number": int(PR_NUMBER), "title": "[DEMO] drop user_id column from users table", "user": {"login": "demo-presenter", "type": "User"}},
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(high_risk).encode(),
    headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"HIGH RISK payload sent (HTTP {res.getcode()})")
print("Agent will: HIGH RISK -> auto-close PR -> create Jira ticket -> send approval email")

## Step 4: Simulate LOW RISK (Docs Update)

In [ ]:
low_risk = {
    "action": "opened",
    "pull_request": {"number": int(PR_NUMBER), "title": "[DEMO] docs: update README", "user": {"login": "demo-presenter", "type": "User"}},
    "repository": {"full_name": REPO},
    "installation": {"id": 12345678},
    "sender": {"login": "demo-presenter", "type": "User"}
}
req = urllib.request.Request(GATEWAY_URL, data=json.dumps(low_risk).encode(),
    headers={'Content-Type': 'application/json', 'X-GitHub-Event': 'pull_request'}, method='POST')
with urllib.request.urlopen(req, timeout=30) as res:
    print(f"LOW RISK payload sent (HTTP {res.getcode()})")
print("Agent will: LOW RISK -> auto-merge PR")

## Step 5: View Live Results
> Run this cell 15 seconds after running cells 2-4.

In [ ]:
import time, re
print("Waiting 15 seconds for agent to finish...")
time.sleep(15)

API = f"https://api.github.com/repos/{REPO}"
H = {"Accept": "application/vnd.github.v3+json", "User-Agent": "PR-Demo"}
if GITHUB_TOKEN:
    H["Authorization"] = f"token {GITHUB_TOKEN}"

def gh(url):
    try:
        req = urllib.request.Request(url, headers=H)
        with urllib.request.urlopen(req, timeout=10) as r:
            return json.loads(r.read())
    except: return None

pr = gh(f"{API}/pulls/{PR_NUMBER}") or gh(f"{API}/issues/{PR_NUMBER}")
if pr:
    merged = pr.get('merged', False)
    state = pr.get('state','?').upper()
    icon = 'MERGED' if merged else ('CLOSED (HIGH RISK - blocked!)' if state=='CLOSED' else 'OPEN')
    print(f"\n=== PR STATUS ===")
    print(f"PR #{PR_NUMBER}: {icon}")
    print(f"URL: {pr.get('html_url','')}")
else:
    print("Could not read PR - is GITHUB_TOKEN set? Is the repo public?")

comments = gh(f"{API}/issues/{PR_NUMBER}/comments") or []
jira = None
print("\n=== AI RISK REPORT ===")
for c in reversed(comments):
    body = c.get('body','')
    if 'RISK' in body.upper():
        print(body[:1000])
        m = re.search(r'([A-Z]+-\d+)', body)
        if m: jira = m.group(1)
        break
else:
    print("No agent comment yet - re-run this cell in a few seconds")

print("\n=== JIRA TICKET ===")
if jira:
    print(f"Created: {jira}")
    print(f"View: https://lmudu95.atlassian.net/browse/{jira}")
else:
    print("No ticket (only for HIGH RISK PRs)")

print("\n=== EMAIL ===")
if pr and pr.get('state')=='closed' and not pr.get('merged'):
    print("Approval email SENT (PR auto-closed = HIGH RISK). Check lmudu95@gmail.com")
elif pr and pr.get('merged'):
    print("No email - PR auto-merged (LOW RISK)")
else:
    print("Still processing - re-run this cell")